# Study API 実践ガイド：Ask-and-Tell・optimize・並列候補・Mixed変数・制約条件

このNotebookでは、**実験候補の提案 → 実験の実行 → 観測結果の登録**というStudy APIの基本操作を、初学者向けに順番に学びます。製造条件や材料配合の探索をイメージできますが、目的関数は動作確認しやすい人工的な式を使います。

**学習目標**
1. `create_study()`、`ask()`、`tell()`の役割とTrialの状態遷移を理解する。
2. `ask_batch(q)`で複数候補を一括提案し、結果を任意の順番で登録する。
3. 整数を含むMixed変数と`CandidateConstraints`を指定する。
4. 候補間制約と、通常の1候補ずつの制約との違いを理解する。
5. `study.optimize()`で目的関数評価を自動実行し、callbackと失敗処理を理解する。
6. 実験履歴をDataFrameとして確認する（pandasがある場合）。

**前提**：robotorchanをインストールしたPython環境。DataFrame表示には任意依存のpandasが必要です。

## 1. 基本用語と実験の流れ

- **SearchSpace**：探索する変数の名前と上下限。例：`x`を0〜1の範囲で探索します。
- **Study**：試行（Trial）を管理し、次に試す候補を提案するオブジェクトです。
- **Trial**：候補パラメータ、試行番号、状態、観測値を持つ記録です。
- **ask**：候補を提案してTrialを`RUNNING`にします。
- **tell**：実験結果をStudyへ返し、Trialを`COMPLETE`または`FAIL`にします。

`WAITING → RUNNING → COMPLETE/FAIL`が基本の状態遷移です。最初の数回は初期候補生成を使うため、GPモデルの学習は不要です。

In [ ]:
import torch

from robotorchan.optim.constraints.contracts import CandidateConstraints
from robotorchan.optim.domains.variable_space import MixedVariableSpace
from robotorchan.study import TrialState, create_study

print("PyTorch:", torch.__version__)

## 2. Ask-and-Tell：1候補を評価する

ここでは最大化問題を考えます。真の目的関数は`f(x) = -(x - 0.7)^2`とし、最適値は`x = 0.7`で0です。実際の用途では、この計算部分を測定装置や実験担当者による評価に置き換えます。

`n_initial_trials=10`として、以下の少数回の試行では初期候補のみを使い、モデル学習の有無による挙動の違いを避けます。

In [ ]:
study = create_study(
    search_space={"x": (0.0, 1.0)},
    direction="maximize",
    n_initial_trials=10,
    seed=42,
)

trial = study.ask()
print("提案された候補:", trial.number, dict(trial.params), trial.state)
assert trial.state is TrialState.RUNNING

x = trial.params["x"]
observed_value = -(x - 0.7) ** 2
finished = study.tell(trial, observed_value)
print("観測結果:", finished.value, finished.state)
assert finished.state is TrialState.COMPLETE

## 3. q-batch：複数候補をまとめて提案する

`ask_batch(3)`は3個の候補を**同時に提案**します。戻り値は3つのTrialからなるタプルです。測定が終了した順に`tell()`を呼べるため、実験装置が複数ある場合にも使えます。

注意：**q-batchは候補をまとめて提案するAPI**です。実験を並列に実行する仕組みそのものは提供しません。外部の実験実行や並列処理は利用者が管理します。

In [ ]:
batch = study.ask_batch(3)
print("候補数:", len(batch))
for candidate in batch:
    print(candidate.number, dict(candidate.params), candidate.state)

for candidate in reversed(batch):
    value = -(candidate.params["x"] - 0.7) ** 2
    study.tell(candidate.number, value)

print("登録された試行数:", len(study.trials))
print("最良パラメータ:", dict(study.best_params))
print("最良目的関数値:", study.best_value)
assert len(study.trials) == 4

## 5. `study.optimize()`：評価ループを自動実行する

`ask()`と`tell()`を自分で繰り返す代わりに、`study.optimize(objective, n_trials=...)`を使うと、**候補提案 → 目的関数の評価 → 結果登録**を指定回数だけ自動実行できます。Optunaに似た使い方ですが、候補提案にはrobotorchanの既存の探索機能を利用します。

- **`objective(trial)`**：引数は`Trial`オブジェクトです。`trial.params`から候補を読み取り、**有限なスカラー値**を返します。
- **`n_trials`**：この呼び出しで新たに実行する試行数です。過去の試行数を含めた総数ではありません。
- **`direction`**：`create_study()`で`"maximize"`または`"minimize"`を指定します。
- **`best_trial` / `best_params` / `best_value`**：完了済みTrialの中で最良の結果を参照します。

以下では、動作を理解しやすいように初期候補の範囲内で4回実行します。GP学習を含む本格的なモデルベース探索は、十分な観測が集まった後の別段階です。

In [ ]:
def objective(trial):
    """候補xを受け取り、最大化したいスカラー評価値を返す。"""
    x = trial.params["x"]
    return -(x - 0.7) ** 2


optimize_study = create_study(
    search_space={"x": (0.0, 1.0)},
    direction="maximize",
    n_initial_trials=10,
    seed=123,
)
optimize_study.optimize(objective, n_trials=4)

print("試行数:", len(optimize_study.trials))
for result in optimize_study.trials:
    print(result.number, dict(result.params), result.value, result.state)

print("最良候補:", dict(optimize_study.best_params))
print("最良評価値:", optimize_study.best_value)
assert len(optimize_study.trials) == 4
assert all(result.state is TrialState.COMPLETE for result in optimize_study.trials)

### 4.1 途中経過を記録する：`callbacks`

`callbacks`には`callback(study, finished_trial)`という2引数の関数を渡します。各試行の終了後に呼ばれるので、ログ出力や外部の実験記録との連携に使えます。

In [ ]:
progress = []


def record_progress(study, finished_trial):
    progress.append((finished_trial.number, finished_trial.value))
    print(
        f"Trial {finished_trial.number}: "
        f"value={finished_trial.value:.4f}, best={study.best_value:.4f}"
    )


callback_study = create_study(
    search_space={"x": (0.0, 1.0)},
    n_initial_trials=10,
    seed=321,
)
callback_study.optimize(objective, n_trials=3, callbacks=[record_progress])
assert len(progress) == 3

### 4.2 評価失敗時の扱い：`catch`

`objective`で例外が発生すると、そのTrialは`FAIL`状態になります。既定では例外を再送出して処理を中断します。`catch=(ValueError,)`のように指定すると、対象の例外を記録して**次の試行へ継続**できます。

`catch`は指定した例外だけを捕捉します。想定外のエラーを隠さないよう、必要な例外型だけを指定してください。

In [ ]:
def sometimes_fails(trial):
    if trial.number == 1:
        raise ValueError("測定失敗を模擬")
    return objective(trial)


failure_study = create_study(
    search_space={"x": (0.0, 1.0)},
    n_initial_trials=10,
    seed=9,
)
failure_study.optimize(sometimes_fails, n_trials=3, catch=(ValueError,))
print([(t.number, t.state.value, t.value) for t in failure_study.trials])
assert [t.state for t in failure_study.trials] == [
    TrialState.COMPLETE,
    TrialState.FAIL,
    TrialState.COMPLETE,
]

## 5. Mixed変数：連続変数と整数変数を組み合わせる

たとえば`x`を連続の配合比率、`count`を整数の処理回数とします。

`MixedVariableSpace`では、**探索変数の並び順**と`integer_dims`のインデックスを一致させます。下の例では`x`が0番、`count`が1番です。探索範囲の`bounds`は`[2, d]`形状で、上段が下限、下段が上限です。

`CandidateConstraints`の不等式は、この例では`count >= 3`を意味します。

In [ ]:
variable_space = MixedVariableSpace(
    bounds=torch.tensor([[0.0, 1.0], [1.0, 4.0]], dtype=torch.double),
    integer_dims=(1,),
)
constraints = CandidateConstraints(
    inequality_constraints=((torch.tensor([1]), torch.tensor([1.0]), 3.0),),
)
mixed_study = create_study(
    search_space={"x": (0.0, 1.0), "count": (1.0, 4.0)},
    variable_space=variable_space,
    constraints=constraints,
    n_initial_trials=10,
    seed=7,
)
mixed_trials = mixed_study.ask_batch(3)
for candidate in mixed_trials:
    print("提案:", candidate.number, dict(candidate.params))
    assert candidate.params["count"] in (3.0, 4.0)
    mixed_study.tell(candidate, candidate.params["x"])

print("Mixed実験の最良候補:", dict(mixed_study.best_params))

## 6. 外部から渡す候補も検証される

実験済み候補や利用者が指定した候補は`add_trial()`で`WAITING`状態として登録できます。Mixed変数では整数性を、制約付きStudyでは候補の可行性を検証します。以下は**例外が発生することが正しい動作**です。

In [ ]:
try:
    mixed_study.add_trial({"x": 0.5, "count": 2.5})
except ValueError as exc:
    print("整数条件に違反:", exc)

try:
    mixed_study.add_trial({"x": 0.5, "count": 2.0})
except ValueError as exc:
    print("制約に違反:", exc)

waiting = mixed_study.add_trial({"x": 0.5, "count": 3.0})
running = mixed_study.start_trial(waiting.number)
mixed_study.tell(running, 0.5)
print("外部候補の登録完了:", mixed_study.get_trial(waiting.number).state)

## 7. 候補間制約：q個の候補をまとめて評価する

単一候補の制約`x >= 0.8`と、**2候補の合計**に関する制約`x₀ + x₁ >= 1.5`は異なります。

下の`indices`は`[[候補番号, 変数番号], ...]`の形状です。`[[0, 0], [1, 0]]`は、0番目と1番目の候補の`x`を参照します。係数`[1, 1]`と右辺`1.5`で`x₀ + x₁ >= 1.5`になります。

候補間制約があるときは、`ask()`で1点ずつ提案するのではなく、制約で指定した候補数に対応する`ask_batch(q)`を使用します。

In [ ]:
joint_constraints = CandidateConstraints(
    inequality_constraints=(
        (
            torch.tensor([[0, 0], [1, 0]]),
            torch.tensor([1.0, 1.0]),
            1.5,
        ),
    ),
)
joint_study = create_study(
    search_space={"x": (0.0, 1.0)},
    constraints=joint_constraints,
    n_initial_trials=10,
    seed=3,
)
joint_trials = joint_study.ask_batch(2)
values = [candidate.params["x"] for candidate in joint_trials]
print("候補:", values, "合計:", sum(values))
assert sum(values) >= 1.5 - 1e-7

for candidate in joint_trials:
    joint_study.tell(candidate, -(candidate.params["x"] - 0.7) ** 2)

## 8. 実験履歴をDataFrameで確認する

`trials_dataframe()`は`number`（試行番号）、`state`（状態）、`value`（目的関数値）、探索変数列を持つ表を返します。`states=(TrialState.COMPLETE,)`で完了済みの試行に絞れます。

pandasをインストールしていない場合は、`pip install "robotorchan[dataframe]"`で追加できます。このセルでは未インストール時に案内を表示します。

In [ ]:
try:
    import pandas as pd
except ImportError:
    print('DataFrame表示には pip install "robotorchan[dataframe]" が必要です。')
else:
    display(study.trials_dataframe())
    display(study.trials_dataframe(states=(TrialState.COMPLETE,)))

## 9. まとめと実利用時の注意

- `ask()` / `ask_batch(q)`で候補を受け取り、外部実験後に`tell()`で観測値を返します。
- `Trial`は状態遷移を持つため、同じ試行を二重に完了させることはできません。
- Mixed変数では変数の順序と`integer_dims`を一致させます。
- 候補間制約は個々の候補ではなく、**q-batch全体**に適用されます。
- 制約付き初期候補は最大**2048回**の棄却サンプリングで生成します。可行領域が狭い場合や可行解が存在しない場合は失敗します。
- `Study.optimize()`は現時点では逐次実行です。モデル学習後のMixed / q-batch探索では、選択したモデル・獲得関数・Optimizerの互換性を別途確認してください。

**次のステップ**：目的関数を実際の測定値に置き換え、`ask()`と`tell()`を外部の実験ワークフローへ組み込んでみてください。